# lunacorder on Google Colab — mineral maps from an M3 scene

Run the cells from top to bottom. The only things you edit are in **Step 1**.

Put these files in one Google Drive folder (default `MyDrive/M3_Project`):
* `<scene>_rfl.img` + `.hdr`, `<scene>_loc.img` + `.hdr`, `<scene>_obs.img` + `.hdr` (from the PDS ODE)
* `ASCIIdata_splib07a.zip` (USGS spectral library, ASCII version)
* optional: `Relab_lunar_mineral_spectra.zip`

In [ ]:
# Fast setup (~10-20 s): clone the code instead of building a pip package.
# numpy, scipy, matplotlib and PyYAML are already in Colab; only rasterio (GeoTIFF output) is added.
import os, sys
REPO = '/content/lunacorder'
BRANCH = 'claude/geospatial-portfolio-projects-qkv0nx'
if not os.path.exists(REPO):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/Dkashkush/Moon-mineralogical-mapper-mineral-mapping {REPO}
else:
    !git -C {REPO} pull -q
sys.path.insert(0, REPO + '/src')
try:
    import rasterio
except ImportError:
    !pip install -q rasterio
import lunacorder
print('lunacorder', lunacorder.__version__, 'ready')

from google.colab import drive
drive.mount('/content/drive')


## Step 1 — your settings

In [ ]:
FOLDER   = '/content/drive/MyDrive/M3_Project'
SCENE_ID = 'm3g20090607t025544_v01'
USGS_ZIP = FOLDER + '/ASCIIdata_splib07a.zip'
RELAB_ZIP = FOLDER + '/Relab_lunar_mineral_spectra.zip'   # set to None if you don't have it
OUT      = FOLDER + '/lunacorder_results'

## Step 2 — build a library matched to *this scene's* bands (run once per scene)

In [ ]:
import os, glob
from lunacorder.pipeline import build_scene_library
from lunacorder.envi import find_header

rfl_hdr = find_header(f'{FOLDER}/{SCENE_ID}_rfl.img')
MINERAL_LIST = FOLDER + '/moon_minerals.txt'   # optional; set to None to keep every USGS spectrum
lib = build_scene_library(rfl_hdr, f'{FOLDER}/{SCENE_ID}_library.npz', usgs=USGS_ZIP,
                          relab=RELAB_ZIP if RELAB_ZIP and os.path.exists(RELAB_ZIP) else None,
                          mineral_list=MINERAL_LIST if MINERAL_LIST and os.path.exists(MINERAL_LIST) else None)


Check which library spectra each mineral will use. **Read this list.** If a mineral picks up the wrong spectra, edit the expert-system YAML (Step 5).

In [ ]:
from lunacorder import ExpertSystem, resolve
from lunacorder.m3 import M3Scene
scene = M3Scene.from_folder(FOLDER, SCENE_ID)
expert = ExpertSystem.builtin()
resolved = resolve(expert, lib, scene.wavelengths, scene.good_bands())
print(resolved.summary())
for i, m in enumerate(expert.materials):
    print('\n' + m.name + ':')
    for r in resolved.references_for(i):
        print('   ', r.name)

## Step 3 — look at the whole strip and pick rows

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from lunacorder.parameters import stretch
lon, lat = scene.read_lonlat()
b = int(np.argmin(np.abs(scene.wavelengths - 1580)))
albedo = scene.rfl.memmap()[::10, b, :] if scene.rfl.interleave == 'bil' else scene.read_reflectance()[::10, :, b]
fig, ax = plt.subplots(1, 2, figsize=(10, 8))
ax[0].imshow(stretch(np.where(albedo > 0, albedo, np.nan)), cmap='gray', aspect='auto',
             extent=[0, albedo.shape[1], scene.rfl.lines, 0])
ax[0].set_title('R1580 (every 10th line)'); ax[0].set_ylabel('row (line)')
ax[1].plot(lat[:, lat.shape[1] // 2], np.arange(lat.shape[0])); ax[1].invert_yaxis()
ax[1].set_xlabel('latitude (°N)'); ax[1].set_title('row → latitude'); ax[1].grid()
plt.show()
print(f'Latitude {np.nanmin(lat):.2f} to {np.nanmax(lat):.2f}, longitude {np.nanmin(lon):.2f} to {np.nanmax(lon):.2f}')

Choose a latitude range; the next cell converts it to rows.

In [ ]:
LAT_SOUTH, LAT_NORTH = 18.0, 19.5
centre = lat[:, lat.shape[1] // 2]
rows_in = np.flatnonzero((centre >= LAT_SOUTH) & (centre <= LAT_NORTH))
ROWS = slice(int(rows_in.min()), int(rows_in.max()) + 1)
print('ROWS =', ROWS, f'({ROWS.stop - ROWS.start} lines)')

## Step 4 — map minerals

In [ ]:
from lunacorder.pipeline import map_scene
result, extras = map_scene(
    FOLDER, SCENE_ID, lib, OUT, expert=expert, rows=ROWS,
    max_incidence=85,     # mask low-sun pixels (deg)
    max_emission=None,    # e.g. 30 to mask oblique views
    max_phase=None,       # e.g. 90
    ensemble=True,        # also run SAM+SID, LSMA and CEM as cross-checks
)


In [ ]:
from IPython.display import Image, display, Markdown
import pandas as pd
for name in ['mineral_map_projected', 'spectra', 'ibd']:
    display(Image(f'{OUT}/{SCENE_ID}_{name}.png', width=900))
display(pd.read_csv(f'{OUT}/{SCENE_ID}_summary.csv'))
display(Markdown('**Cross-check: how often SAM+SID, LSMA and CEM agree with feature fitting**'))
display(pd.read_csv(f'{OUT}/{SCENE_ID}_crosscheck.csv'))
display(Markdown(open(f'{OUT}/{SCENE_ID}_methods.md').read()))


## Step 5 (optional) — tune the expert system

Copy the built-in rules to Drive, edit thresholds or reference patterns, then re-run Step 4 with `expert=ExpertSystem.from_yaml(...)`.
Report whatever values you use in your methods section.

In [ ]:
from importlib import resources
import shutil
src = resources.files('lunacorder.data').joinpath('lunar_m3.yaml')
shutil.copy(src, f'{FOLDER}/my_lunar_rules.yaml')
print('Edit', f'{FOLDER}/my_lunar_rules.yaml', 'then use expert=ExpertSystem.from_yaml(that path)')

## Optional — share a small test area

This writes a ~18 MB copy of 150 lines of the scene (reflectance + LOC + OBS, original headers)
plus the convolved library to `MyDrive/M3_Project/share_subset/`. That is small enough to share
for debugging or for validating the method on real data. Pick rows that contain something interesting.

In [ ]:
from lunacorder.m3 import subset_scene
import shutil
SHARE = FOLDER + '/share_subset'
SHARE_ROWS = slice(ROWS.start, ROWS.start + 150)
subset_scene(scene, SHARE, SHARE_ROWS)
shutil.copy(f'{FOLDER}/{SCENE_ID}_library.npz', SHARE)
for f in sorted(os.listdir(SHARE)):
    print(f'{f:45s} {os.path.getsize(os.path.join(SHARE, f)) / 1e6:7.1f} MB')